# 14 — RQ2 blind audit construction

Builds the 240-comment blind transfer audit: retains the 180 completed annotations from the three comparison communities, drops the obsolete provisional manosphere set, and samples a fresh 60 manosphere-adjacent comments using the prediction-pattern-enriched procedure (seed 42). After the new 60 are hand-labelled, the completed labels are merged with the hidden key and validated (text match, hierarchy, community composition, sampling weights).

Inputs and outputs live under `outputs/rq2/`. Requires the scored corpus from notebook 13.


In [ ]:
# ============================================================
# RECONSTRUCT FINAL RQ2 AUDIT
#
# Retains the exact 180 completed annotations from the three
# unchanged communities.
#
# Removes the obsolete 60-comment provisional manosphere audit.
#
# Samples a new 60-comment manosphere audit using the EXACT
# original prediction-pattern-enriched procedure.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

# Input files live in outputs/rq2/; all generated
# audit files are written permanently to Google Drive.

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "outputs" / "rq2"

FINAL_RQ2_PATH = (
    DATA_DIR
    / "rq2_FINAL_primary_12mo_SCORED.csv"
)

OLD_LABELLED_PATH = (
    DATA_DIR
    / "rq2_blind_validation_240_labelled.csv"
)

OLD_KEY_PATH = (
    DATA_DIR
    / "rq2_blind_validation_240_KEY_DO_NOT_OPEN.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rq2"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


NEW_BLIND_60_PATH = (
    OUTPUT_DIR
    / "rq2_final_manosphere_blind_60_TO_LABEL.csv"
)

PARTIAL_BLIND_240_PATH = (
    OUTPUT_DIR
    / "rq2_final_audit_240_PARTIALLY_LABELLED.csv"
)

FINAL_KEY_PATH = (
    OUTPUT_DIR
    / "rq2_final_audit_240_KEY_DO_NOT_OPEN.csv"
)

EXCLUDED_OLD_60_PATH = (
    OUTPUT_DIR
    / "rq2_old_provisional_manosphere_60_EXCLUDED.csv"
)


# ============================================================
# 2. LOAD FILES
# ============================================================

required_input_paths = [
    FINAL_RQ2_PATH,
    OLD_LABELLED_PATH,
    OLD_KEY_PATH,
]

missing_input_paths = [
    path
    for path in required_input_paths
    if not path.exists()
]

if missing_input_paths:

    missing_text = "\n".join(
        str(path)
        for path in missing_input_paths
    )

    raise FileNotFoundError(
        "Place the following source files in outputs/rq2/ "
        "and rerun the whole cell:\n"
        f"{missing_text}"
    )

final_rq2 = pd.read_csv(
    FINAL_RQ2_PATH,
    low_memory=False,
    dtype={
        "comment_id": "string"
    }
)

old_manual = pd.read_csv(
    OLD_LABELLED_PATH,
    encoding="utf-8-sig"
)

old_key = pd.read_csv(
    OLD_KEY_PATH,
    low_memory=False,
    dtype={
        "comment_id": "string",
        "prediction_pattern": "string"
    }
)

old_key["prediction_pattern"] = (
    old_key["prediction_pattern"]
    .astype("string")
    .str.zfill(3)
)


print("=" * 72)
print("FINAL RQ2 AUDIT RECONSTRUCTION")
print("=" * 72)

print("\nFinal RQ2 rows:", len(final_rq2))
print("Old manual rows:", len(old_manual))
print("Old key rows:", len(old_key))


# ============================================================
# 3. BASIC CHECKS
# ============================================================

assert len(final_rq2) == 22326
assert len(old_manual) == 240
assert len(old_key) == 240

assert old_manual["audit_id"].is_unique
assert old_key["audit_id"].is_unique
assert final_rq2["comment_id"].is_unique

assert (
    set(old_manual["audit_id"])
    ==
    set(old_key["audit_id"])
)

required_final_columns = {
    "comment_id",
    "community_group",
    "channel_name",
    "channel_handle",
    "video_id",
    "text_clean",
    "pred_1A",
    "pred_1B",
    "pred_2",
    "prob_1A",
    "prob_1B",
    "prob_2"
}

missing = (
    required_final_columns
    -
    set(final_rq2.columns)
)

if missing:

    raise ValueError(
        f"Final RQ2 file is missing: {missing}"
    )


# ============================================================
# 4. IDENTIFY RETAINED AND OBSOLETE OLD ROWS
# ============================================================

final_comment_ids = set(
    final_rq2["comment_id"]
)

retained_key = old_key[
    old_key["comment_id"].isin(
        final_comment_ids
    )
].copy()

obsolete_key = old_key[
    ~old_key["comment_id"].isin(
        final_comment_ids
    )
].copy()


assert len(retained_key) == 180
assert len(obsolete_key) == 60

assert set(
    retained_key["community_group"]
) == {
    "mainstream_self_improvement",
    "male_oriented_wellbeing",
    "unrelated_control"
}

assert set(
    obsolete_key["community_group"]
) == {
    "manosphere_adjacent"
}


print("\nRetained old audit rows:", len(retained_key))
print("Obsolete manosphere rows:", len(obsolete_key))

print("\nRetained rows by community:")

print(
    retained_key[
        "community_group"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 5. VERIFY RETAINED ROWS AGAINST FINAL RQ2
# ============================================================

verification_columns = [
    "comment_id",
    "community_group",
    "pred_1A",
    "pred_1B",
    "pred_2"
]

verification = retained_key[
    verification_columns
].merge(
    final_rq2[
        verification_columns
    ],
    on="comment_id",
    how="inner",
    validate="one_to_one",
    suffixes=(
        "_old",
        "_final"
    )
)

assert len(verification) == 180

assert (
    verification["community_group_old"]
    ==
    verification["community_group_final"]
).all()

for label in [
    "pred_1A",
    "pred_1B",
    "pred_2"
]:

    assert (
        verification[f"{label}_old"]
        ==
        verification[f"{label}_final"]
    ).all()


print(
    "\nVerified: all retained comments have "
    "unchanged communities and predictions."
)


# ============================================================
# 6. RETAIN THEIR COMPLETED MANUAL LABELS
# ============================================================

manual_columns = [
    "audit_id",
    "text_clean",
    "manual_1A",
    "manual_1B",
    "manual_2",
    "notes"
]

retained_manual = old_manual[
    old_manual["audit_id"].isin(
        retained_key["audit_id"]
    )
][manual_columns].copy()

assert len(retained_manual) == 180

assert retained_manual[
    [
        "manual_1A",
        "manual_1B",
        "manual_2"
    ]
].notna().all().all()


# ============================================================
# 7. PRESERVE THE EXCLUDED PROVISIONAL 60
# ============================================================

excluded_old_60 = old_manual.merge(
    obsolete_key,
    on="audit_id",
    how="inner",
    validate="one_to_one",
    suffixes=(
        "_manual",
        "_key"
    )
)

assert len(excluded_old_60) == 60

excluded_old_60.to_csv(
    EXCLUDED_OLD_60_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8. SELECT FINAL MANOSPHERE CORPUS
# ============================================================

manosphere = final_rq2[
    final_rq2["community_group"]
    ==
    "manosphere_adjacent"
].copy()

assert len(manosphere) == 7233

assert set(
    manosphere[
        "channel_handle"
    ].dropna()
) == {
    "@sandmanmgtow",
    "@manguide",
    "@thechowtimepod",
    "@itscomplicatedchannel"
}


# ============================================================
# 9. CREATE JOINT PREDICTION PATTERN
# ============================================================

manosphere["prediction_pattern"] = (
    manosphere[
        [
            "pred_1A",
            "pred_1B",
            "pred_2"
        ]
    ]
    .astype(int)
    .astype(str)
    .agg("".join, axis=1)
)


print("\nFinal manosphere prediction patterns:")

print(
    manosphere[
        "prediction_pattern"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 10. REPRODUCE THE ORIGINAL SAMPLING PROCEDURE
#
# Original procedure:
# - 60 comments per community
# - up to 5 from every non-000 pattern
# - fill remaining positions with 000 comments
# - seed 42
# ============================================================

TARGET_N = 60
MAX_PER_POSITIVE_PATTERN = 5
SEED = 42

selected_indices = []

positive_patterns = sorted(
    pattern
    for pattern
    in manosphere[
        "prediction_pattern"
    ].unique()
    if pattern != "000"
)


for pattern in positive_patterns:

    pool = manosphere[
        manosphere[
            "prediction_pattern"
        ]
        ==
        pattern
    ]

    n_take = min(
        MAX_PER_POSITIVE_PATTERN,
        len(pool)
    )

    chosen = pool.sample(
        n=n_take,
        random_state=SEED
    )

    selected_indices.extend(
        chosen.index.tolist()
    )


remaining = (
    TARGET_N
    -
    len(selected_indices)
)

if remaining < 0:

    raise ValueError(
        "Positive-pattern sample exceeded 60 rows."
    )


negative_pool = manosphere[
    manosphere[
        "prediction_pattern"
    ]
    ==
    "000"
]

negative_sample = negative_pool.sample(
    n=remaining,
    random_state=SEED
)

selected_indices.extend(
    negative_sample.index.tolist()
)


new_sample = (
    manosphere
    .loc[selected_indices]
    .copy()
)

assert len(new_sample) == 60
assert new_sample["comment_id"].nunique() == 60

assert set(
    new_sample["comment_id"]
).isdisjoint(
    set(retained_key["comment_id"])
)


# ============================================================
# 11. RECALCULATE WEIGHTS FOR FINAL MANOSPHERE CORPUS
# ============================================================

population_counts = (
    manosphere.groupby(
        [
            "community_group",
            "prediction_pattern"
        ]
    )
    .size()
    .rename(
        "stratum_population_n"
    )
    .reset_index()
)

sample_counts = (
    new_sample.groupby(
        [
            "community_group",
            "prediction_pattern"
        ]
    )
    .size()
    .rename(
        "stratum_sample_n"
    )
    .reset_index()
)

new_sample = (
    new_sample
    .drop(
        columns=[
            "stratum_population_n",
            "stratum_sample_n",
            "sampling_weight"
        ],
        errors="ignore"
    )
    .merge(
        population_counts,
        on=[
            "community_group",
            "prediction_pattern"
        ],
        how="left"
    )
    .merge(
        sample_counts,
        on=[
            "community_group",
            "prediction_pattern"
        ],
        how="left"
    )
)

new_sample["sampling_weight"] = (
    new_sample[
        "stratum_population_n"
    ]
    /
    new_sample[
        "stratum_sample_n"
    ]
)

assert np.isclose(
    new_sample[
        "sampling_weight"
    ].sum(),
    len(manosphere)
)


# ============================================================
# 12. SHUFFLE AND CREATE NEW BLIND IDS
# ============================================================

new_sample = (
    new_sample
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

new_sample["audit_id"] = [
    f"FINAL_M_{number:03d}"
    for number in range(
        1,
        61
    )
]


# ============================================================
# 13. CREATE NEW HIDDEN KEY
# ============================================================

key_columns = [
    "audit_id",
    "comment_id",
    "community_group",
    "channel_name",
    "video_id",
    "prediction_pattern",
    "pred_1A",
    "pred_1B",
    "pred_2",
    "prob_1A",
    "prob_1B",
    "prob_2",
    "stratum_population_n",
    "stratum_sample_n",
    "sampling_weight",
    "text_clean"
]

new_key = new_sample[
    key_columns
].copy()

final_key = pd.concat(
    [
        retained_key[
            key_columns
        ],
        new_key
    ],
    ignore_index=True
)

assert len(final_key) == 240
assert final_key["audit_id"].is_unique
assert final_key["comment_id"].is_unique

assert (
    final_key[
        "community_group"
    ]
    .value_counts()
    .eq(60)
    .all()
)

final_key.to_csv(
    FINAL_KEY_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. CREATE 60-ROW BLIND FILE TO LABEL
# ============================================================

new_blind = new_sample[
    [
        "audit_id",
        "text_clean"
    ]
].copy()

# Use genuine missing values. Empty strings are not counted by
# Series.isna(), which caused the original assertion to fail.
new_blind["manual_1A"] = pd.NA
new_blind["manual_1B"] = pd.NA
new_blind["manual_2"] = pd.NA
new_blind["notes"] = pd.NA

new_blind.to_csv(
    NEW_BLIND_60_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. CREATE PARTIALLY COMPLETED FINAL 240 FILE
#
# The original 180 labels remain completed.
# Only the 60 FINAL_M rows are blank.
# ============================================================

partial_final = pd.concat(
    [
        retained_manual,
        new_blind
    ],
    ignore_index=True
)

partial_final = (
    partial_final
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

assert len(partial_final) == 240
assert partial_final["audit_id"].is_unique

label_columns = [
    "manual_1A",
    "manual_1B",
    "manual_2"
]

for column in label_columns:

    blank_mask = (
        partial_final[column]
        .replace(
            r"^\s*$",
            pd.NA,
            regex=True
        )
        .isna()
    )

    blank_count = int(
        blank_mask.sum()
    )

    print(
        column,
        "| blank rows:",
        blank_count
    )

    assert blank_count == 60

partial_final.to_csv(
    PARTIAL_BLIND_240_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 16. FINAL QC
# ============================================================

print("\n")
print("=" * 72)
print("FINAL AUDIT SAMPLE CREATED")
print("=" * 72)

print("\nFinal community counts:")

print(
    final_key[
        "community_group"
    ]
    .value_counts()
    .sort_index()
)

print("\nNew manosphere sample by prediction pattern:")

print(
    new_key[
        "prediction_pattern"
    ]
    .value_counts()
    .sort_index()
)

print("\nNew manosphere sample by final channel:")

print(
    new_sample[
        "channel_handle"
    ]
    .value_counts()
)

print("\nNew sample predicted positives:")

print(
    new_sample[
        [
            "pred_1A",
            "pred_1B",
            "pred_2"
        ]
    ].sum()
)

print("\nRetained completed labels:", len(retained_manual))
print("New labels required:", len(new_blind))
print("Final audit size:", len(final_key))

print("\nLABEL THIS FILE:")
print(NEW_BLIND_60_PATH)

print("\nOPTIONAL COMBINED PARTIAL FILE:")
print(PARTIAL_BLIND_240_PATH)

print("\nDO NOT OPEN WHILE LABELLING:")
print(FINAL_KEY_PATH)

print("\nEXCLUDED OLD MANOSPHERE ROWS:")
print(EXCLUDED_OLD_60_PATH)

print("\nDONE.")


# ============================================================
# 17. VERIFY THAT EVERY OUTPUT WAS WRITTEN TO DRIVE
# ============================================================

output_paths = [
    NEW_BLIND_60_PATH,
    PARTIAL_BLIND_240_PATH,
    FINAL_KEY_PATH,
    EXCLUDED_OLD_60_PATH,
]

print("\n")
print("=" * 72)
print("PERSISTENT OUTPUT CHECK")
print("=" * 72)

for path in output_paths:

    if not path.exists():
        raise FileNotFoundError(
            f"Expected output was not created: {path}"
        )

    print(
        path.name,
        "| saved:",
        path.exists(),
        "| bytes:",
        path.stat().st_size
    )

print("\nAll four files are saved permanently in:")
print(OUTPUT_DIR)


In [ ]:
# ============================================================
# FIX AND SAVE OPTIONAL PARTIALLY LABELLED 240-ROW FILE
# ============================================================

import pandas as pd


partial_final = pd.concat(
    [
        retained_manual,
        new_blind
    ],
    ignore_index=True
)

partial_final = (
    partial_final
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)


assert len(partial_final) == 240
assert partial_final["audit_id"].is_unique


label_columns = [
    "manual_1A",
    "manual_1B",
    "manual_2"
]


for column in label_columns:

    blank_mask = (
        partial_final[column]
        .replace(
            r"^\s*$",
            pd.NA,
            regex=True
        )
        .isna()
    )

    blank_count = int(
        blank_mask.sum()
    )

    print(
        column,
        "| blank rows:",
        blank_count
    )

    assert blank_count == 60


partial_final.to_csv(
    PARTIAL_BLIND_240_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\nSaved:")
print(PARTIAL_BLIND_240_PATH)

## Manual annotation handoff

Cell 0 above writes the blind 60-comment manosphere file (`rq2_final_manosphere_blind_60_TO_LABEL.csv`) and the partially-labelled 240 file. The 60 new comments are labelled **by hand** by the annotator, offline, without access to the model predictions (the key is held separately). Place the completed file at `outputs/rq2/rq2_final_audit_240_labelled.csv`, then run the merge cell below.

*(The original Colab workflow also uploaded the completed file back into the session and listed temporary files; those manual/recovery steps are not part of the reproducible pipeline and have been removed.)*


In [ ]:
# ============================================================
# MERGE COMPLETED RQ2 AUDIT WITH HIDDEN KEY
# ============================================================

from pathlib import Path
import unicodedata

import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LABELLED_PATH = PROJECT_ROOT / "outputs" / "rq2" / "rq2_final_audit_240_labelled.csv"

KEY_PATH = PROJECT_ROOT / "outputs" / "rq2" / "rq2_final_audit_240_KEY_DO_NOT_OPEN.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rq2"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_LABELLED_PATH = (
    OUTPUT_DIR
    / "rq2_final_audit_240_labelled.csv"
)

FINAL_KEY_PATH = (
    OUTPUT_DIR
    / "rq2_final_audit_240_KEY_DO_NOT_OPEN.csv"
)

MERGED_PATH = (
    OUTPUT_DIR
    / "rq2_final_audit_240_MERGED.csv"
)


# ============================================================
# 2. CHECK INPUT FILES
# ============================================================

for path in [LABELLED_PATH, KEY_PATH]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


# ============================================================
# 3. LOAD FILES
# ============================================================

labelled = pd.read_csv(
    LABELLED_PATH,
    encoding="utf-8-sig"
)

key = pd.read_csv(
    KEY_PATH,
    encoding="utf-8-sig"
)


# Clean column names
labelled.columns = (
    labelled.columns
    .astype(str)
    .str.strip()
)

key.columns = (
    key.columns
    .astype(str)
    .str.strip()
)


# Remove accidental index columns
labelled = labelled.loc[
    :,
    ~labelled.columns.str.startswith("Unnamed:")
].copy()

key = key.loc[
    :,
    ~key.columns.str.startswith("Unnamed:")
].copy()


print("=" * 72)
print("FILES LOADED")
print("=" * 72)

print("\nCompleted annotations:", labelled.shape)
print("Hidden key:", key.shape)


# ============================================================
# 4. REQUIRED COLUMNS
# ============================================================

manual_columns = [
    "manual_1A",
    "manual_1B",
    "manual_2",
]

required_labelled = {
    "audit_id",
    "text_clean",
    *manual_columns,
}

required_key = {
    "audit_id",
    "text_clean",
    "community_group",
    "prediction_pattern",
    "pred_1A",
    "pred_1B",
    "pred_2",
    "sampling_weight",
}

missing_labelled = (
    required_labelled
    - set(labelled.columns)
)

missing_key = (
    required_key
    - set(key.columns)
)

if missing_labelled:

    raise ValueError(
        "Completed annotation file is missing columns:\n"
        f"{sorted(missing_labelled)}"
    )

if missing_key:

    raise ValueError(
        "Hidden key is missing columns:\n"
        f"{sorted(missing_key)}"
    )


if "notes" not in labelled.columns:
    labelled["notes"] = pd.NA


# ============================================================
# 5. VALIDATE AUDIT IDS
# ============================================================

labelled["audit_id"] = (
    labelled["audit_id"]
    .astype(str)
    .str.strip()
)

key["audit_id"] = (
    key["audit_id"]
    .astype(str)
    .str.strip()
)


assert len(labelled) == 240, (
    f"Expected 240 labelled rows; found {len(labelled)}."
)

assert len(key) == 240, (
    f"Expected 240 key rows; found {len(key)}."
)

assert labelled["audit_id"].is_unique, (
    "Duplicate audit IDs found in labelled file."
)

assert key["audit_id"].is_unique, (
    "Duplicate audit IDs found in key."
)

assert set(labelled["audit_id"]) == set(key["audit_id"]), (
    "The audit IDs in the completed file do not exactly match the key."
)


# ============================================================
# 6. VALIDATE MANUAL LABELS
# ============================================================

for column in manual_columns:

    labelled[column] = pd.to_numeric(
        labelled[column],
        errors="coerce"
    )

    if labelled[column].isna().any():

        bad_rows = labelled.loc[
            labelled[column].isna(),
            ["audit_id", column]
        ]

        raise ValueError(
            f"{column} contains missing or invalid values.\n"
            f"{bad_rows.head(10)}"
        )

    if not labelled[column].isin([0, 1]).all():

        raise ValueError(
            f"{column} contains values other than 0 and 1."
        )

    labelled[column] = (
        labelled[column]
        .astype(int)
    )


# Check the manual hierarchy
manual_hierarchy_violations = labelled.loc[
    (labelled["manual_1B"] == 1)
    & (labelled["manual_1A"] == 0)
]

assert manual_hierarchy_violations.empty, (
    "Manual hierarchy violation found: "
    "manual_1B=1 while manual_1A=0."
)


# ============================================================
# 7. VERIFY THAT THE TEXT MATCHES THE KEY
# ============================================================

def normalise_text(value):

    if pd.isna(value):
        return ""

    value = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    return " ".join(
        value.split()
    )


text_check = key[
    ["audit_id", "text_clean"]
].merge(
    labelled[
        ["audit_id", "text_clean"]
    ],
    on="audit_id",
    how="left",
    validate="one_to_one",
    suffixes=("_key", "_manual")
)


text_check["text_matches"] = (
    text_check["text_clean_key"]
    .map(normalise_text)
    ==
    text_check["text_clean_manual"]
    .map(normalise_text)
)


if not text_check["text_matches"].all():

    mismatches = text_check.loc[
        ~text_check["text_matches"],
        [
            "audit_id",
            "text_clean_key",
            "text_clean_manual",
        ]
    ]

    raise ValueError(
        "Text does not match the hidden key for "
        f"{len(mismatches)} rows.\n\n"
        f"{mismatches.head()}"
    )


# ============================================================
# 8. VERIFY MODEL PREDICTION PATTERNS
# ============================================================

for column in ["pred_1A", "pred_1B", "pred_2"]:

    key[column] = pd.to_numeric(
        key[column],
        errors="raise"
    ).astype(int)

    assert key[column].isin([0, 1]).all()


expected_pattern = (
    key["pred_1A"].astype(str)
    + key["pred_1B"].astype(str)
    + key["pred_2"].astype(str)
)

stored_pattern = (
    key["prediction_pattern"]
    .astype(str)
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
    .str.zfill(3)
)

assert (
    expected_pattern == stored_pattern
).all(), (
    "At least one stored prediction pattern "
    "does not match pred_1A/pred_1B/pred_2."
)

key["prediction_pattern"] = stored_pattern


# ============================================================
# 9. VERIFY COMMUNITY COMPOSITION AND WEIGHTS
# ============================================================

community_counts = (
    key["community_group"]
    .value_counts()
    .sort_index()
)

assert len(community_counts) == 4, (
    "Expected four community groups."
)

assert (community_counts == 60).all(), (
    "Expected exactly 60 audit comments per community.\n"
    f"{community_counts}"
)


key["sampling_weight"] = pd.to_numeric(
    key["sampling_weight"],
    errors="raise"
)

assert (
    key["sampling_weight"] > 0
).all(), (
    "Sampling weights must all be positive."
)


# ============================================================
# 10. MERGE MANUAL LABELS INTO THE KEY
# ============================================================

# Remove stale manual columns if this cell is rerun
key_clean = key.drop(
    columns=[
        *manual_columns,
        "notes",
    ],
    errors="ignore"
)


merged = key_clean.merge(
    labelled[
        [
            "audit_id",
            *manual_columns,
            "notes",
        ]
    ],
    on="audit_id",
    how="left",
    validate="one_to_one"
)


assert len(merged) == 240
assert merged["audit_id"].is_unique
assert merged[manual_columns].notna().all().all()


# ============================================================
# 11. SAVE PERMANENT COPIES
# ============================================================

labelled.to_csv(
    FINAL_LABELLED_PATH,
    index=False,
    encoding="utf-8-sig"
)

key.to_csv(
    FINAL_KEY_PATH,
    index=False,
    encoding="utf-8-sig"
)

merged.to_csv(
    MERGED_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n" + "=" * 72)
print("FINAL RQ2 AUDIT MERGE COMPLETE")
print("=" * 72)

print("\nRows:", len(merged))
print("Unique audit IDs:", merged["audit_id"].nunique())
print("Text mismatches:", int((~text_check["text_matches"]).sum()))
print("Manual hierarchy violations:", len(manual_hierarchy_violations))

print("\nCommunity composition:")
print(community_counts)

print("\nManual positive counts:")
print(
    merged[
        manual_columns
    ].sum()
)

print("\nManual positives by community:")
print(
    merged.groupby(
        "community_group"
    )[manual_columns]
    .sum()
)

print("\nPrediction-pattern composition:")
print(
    pd.crosstab(
        merged["community_group"],
        merged["prediction_pattern"]
    )
)

print("\nSaved permanently:")
print(FINAL_LABELLED_PATH)
print(FINAL_KEY_PATH)
print(MERGED_PATH)

print("\nREADY FOR WEIGHTED AUDIT ANALYSIS.")